# Receipt VLM — merge a checkpoint & run it in the OCR pipeline

Takes **one training checkpoint** you produced (e.g. a `phase2_best.pt` / `phase2_epoch*.pt`
you downloaded from Kaggle), merges LoRA into the backbone to make an inference-ready
`receipt_vlm_500m_merged.pt`, and runs it through the real `receipt_ocr` pipeline on the
**same hand-labelled real receipts the rest of the codebase evaluates on** — comparing each
prediction against its ground-truth label and printing the standard acceptance metrics.

Run it locally from anywhere inside the repo (the `receipt_ocr` / `receipt_vlm` packages are
already editable-installed in the dev venv). Edit **cell 0**, then *Run All*.

## 0. Configuration — edit then run

In [ ]:
# Path to the checkpoint you downloaded from Kaggle (REQUIRED), e.g. a phase-2 snapshot:
CHECKPOINT = r"checkpoints/phase2_best.pt"

# Where to write the merged inference model. "" -> next to the checkpoint.
MERGED_OUT = ""

# Real-life labelled set (same images + labels used by scripts/evaluate.py).
# Leave IMAGES_DIR / LABELS_DIR = "" to use the repo's canonical locations.
IMAGES_DIR = ""
LABELS_DIR = ""
SPLIT = "test"      # "test" (hand-reviewed held-out) | "val" | "train" | "" for all labelled
MAX_IMAGES = 0      # 0 = every image in the split

## 1. Locate the packages
Walks up from the working dir to find `dev_ocr/vlm_training`, and puts `receipt_ocr` /
`receipt_vlm` on the path (a fallback in case they aren't installed in this kernel).

In [ ]:
import sys
from pathlib import Path

def _find_train_pkg() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "scripts" / "export_checkpoint.py").is_file():
            return base
        cand = base / "dev_ocr" / "vlm_training"
        if (cand / "scripts" / "export_checkpoint.py").is_file():
            return cand
    raise RuntimeError("Couldn't locate dev_ocr/vlm_training -- set TRAIN_PKG manually")

TRAIN_PKG = _find_train_pkg()          # .../dev_ocr/vlm_training
DEV_OCR = TRAIN_PKG.parent             # .../dev_ocr
for p in (str(DEV_OCR / "src"), str(TRAIN_PKG)):   # receipt_ocr (src layout) + receipt_vlm
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Train package:", TRAIN_PKG)
print("Device       :", DEVICE,
      "-", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only (slow generation)")

## 2. Merge the checkpoint
Runs `scripts/export_checkpoint.py` to fold the LoRA adapters into the frozen backbone and
write a single inference-ready `.pt`.

In [ ]:
import subprocess, sys
from pathlib import Path

ckpt = Path(CHECKPOINT)
assert ckpt.is_file(), f"CHECKPOINT not found: {ckpt.resolve()}"
merged = Path(MERGED_OUT) if MERGED_OUT else ckpt.with_name("receipt_vlm_500m_merged.pt")

subprocess.check_call([
    sys.executable, str(TRAIN_PKG / "scripts" / "export_checkpoint.py"),
    "--checkpoint", str(ckpt), "--output", str(merged),
])
MERGED = str(merged.resolve())
print("\nMerged ->", MERGED, f"({merged.stat().st_size/1e6:.0f} MB)")
print("Built from:", ckpt.name)

## 3. Run the OCR pipeline on the labelled real receipts
Loads the same real-life labelled receipts as `scripts/evaluate.py` (via `load_real_samples`),
runs each through `receipt_ocr.extract_receipt()` with the merged model as the VLM provider,
and prints the **prediction next to its ground-truth label** so you can eyeball quality.

In [ ]:
import os, json, time
from pathlib import Path

os.environ.update({
    "RECEIPT_OCR_BACKEND":    "vlm",
    "RECEIPT_VLM_MODEL":      "receipt-vlm-500m",
    "RECEIPT_VLM_MODE":       "json",
    "RECEIPT_VLM_MODEL_PATH": MERGED,
})

from receipt_ocr import extract_receipt                       # full pipeline (VLM provider)
from receipt_vlm.data.real_photos import load_real_samples    # same loader evaluate.py uses
from receipt_vlm.data.schema import ticket_from_dict

images_dir = Path(IMAGES_DIR) if IMAGES_DIR else DEV_OCR / "data" / "raw" / "images_tickets_caisse"
labels_dir = Path(LABELS_DIR) if LABELS_DIR else TRAIN_PKG / "data" / "real_labels"
samples = load_real_samples(images_dir, labels_dir, split=(SPLIT or None))
assert samples, f"No labelled samples for split={SPLIT!r} under {labels_dir}"
if MAX_IMAGES:
    samples = samples[:MAX_IMAGES]
print(f"Testing on {len(samples)} labelled '{SPLIT or 'all'}' receipts from {images_dir}\n")

predictions, golds, n_valid = [], [], 0
for s in samples:
    t = time.time()
    try:
        out = extract_receipt(str(s.image))
        pred = ticket_from_dict(out)
        n_valid += 1
    except Exception as e:
        out = {"error": f"{type(e).__name__}: {e}"}
        pred = ticket_from_dict({})
    predictions.append(pred)
    golds.append(s.ticket)
    print(f"=== {Path(s.image).name}  ({time.time()-t:.1f}s) ===")
    print("  predicted:", json.dumps(out, ensure_ascii=False)[:700])
    print("  gold     :", json.dumps(s.ticket.to_dict(), ensure_ascii=False)[:700], "\n")

print(f"Valid JSON: {n_valid}/{len(samples)}")

## 4. Acceptance metrics (vs ground truth)
The same metrics `scripts/evaluate.py` reports — computed here on the pipeline's output so the
numbers reflect the *deployed* path, not just the raw model.

In [ ]:
from receipt_vlm.utils.metrics import evaluate_tickets

metrics = evaluate_tickets(predictions, golds)
TARGETS = [
    ("field_f1",       "Field F1",         "> 0.85"),
    ("product_recall", "Product recall",   "> 0.90"),
    ("price_mae",      "Price MAE (EUR)",  "< 0.05"),
    ("date_accuracy",  "Date exact match", "> 0.90"),
    ("anls",           "ANLS",             "> 0.80"),
]
print(f"{'Metric':22} {'Target':10} {'receipt-vlm-500m'}")
print("-" * 50)
for key, label, target in TARGETS:
    if key in metrics:
        print(f"{label:22} {target:10} {metrics[key]:.3f}")
print(f"\nn = {len(predictions)} labelled receipts  |  checkpoint: {Path(CHECKPOINT).name}")

## 5. Deploy it
The merged model is a single `.pt`. To run it in your inference infrastructure (e.g. the OCR
worker), point the same env vars at it:

```bash
RECEIPT_OCR_BACKEND=vlm
RECEIPT_VLM_MODEL=receipt-vlm-500m
RECEIPT_VLM_MODE=json
RECEIPT_VLM_MODEL_PATH=/path/to/receipt_vlm_500m_merged.pt
```

PowerShell equivalent:
```powershell
$env:RECEIPT_OCR_BACKEND    = "vlm"
$env:RECEIPT_VLM_MODEL      = "receipt-vlm-500m"
$env:RECEIPT_VLM_MODE       = "json"
$env:RECEIPT_VLM_MODEL_PATH = "C:\path\to\receipt_vlm_500m_merged.pt"
```